# Titanic — Preprocessing Pipeline
**Dataset :** Titanic (train.csv)  
**Sequence :** Duplicates → Missing values → Feature engineering → Outliers → Scaling → Encoding → Age transformation


## Setup — imports and data loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# Load dataset
df = pd.read_csv("train.csv")

print("Shape:", df.shape)
df.head()


---
## Exercise 1 — Duplicate Detection and Removal

**Objective :** Identify and remove duplicate rows from the Titanic dataset.


In [ ]:
# Count rows before deduplication
rows_before = df.shape[0]
print(f"Rows before deduplication : {rows_before}")

# Detect duplicates across all columns
n_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows  : {n_duplicates}")

# If duplicates exist, display them
if n_duplicates > 0:
    print("\nDuplicate rows:")
    display(df[df.duplicated(keep=False)])

# Remove duplicate rows, keeping the first occurrence
df = df.drop_duplicates()

# Verify
rows_after = df.shape[0]
print(f"\nRows after deduplication  : {rows_after}")
print(f"Rows removed              : {rows_before - rows_after}")

# Explanation:
# The Titanic dataset is already clean — no duplicates are expected.
# However, applying this step is a systematic good practice in any
# preprocessing pipeline. duplicated() returns a boolean Series;
# drop_duplicates() keeps only the first occurrence by default.


---
## Exercise 2 — Handling Missing Values

**Strategies applied :**
- `Age`      : imputation by the **median** (robust to outliers)
- `Embarked` : imputation by the **mode** (most frequent value)
- `Cabin`    : **dropped** (more than 77 % of values are missing; not recoverable)
- `Fare`     : imputation by the **median** (one missing value in test set)


In [ ]:
# Step 1 — Identify missing values
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Missing values (%):")
print((df.isnull().sum() / len(df) * 100).round(2))


In [ ]:
# Step 2 — Visualise missing data
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap="viridis")
plt.title("Missing values heatmap (yellow = missing)")
plt.tight_layout()
plt.show()


In [ ]:
# Strategy A — Drop 'Cabin' : too many missing values (> 77 %)
df.drop(columns=["Cabin"], inplace=True)
print("'Cabin' column dropped.")

# Strategy B — Impute 'Age' with the median using SimpleImputer
# Median is preferred over mean because Age contains outliers.
imputer_age = SimpleImputer(strategy="median")
df["Age"] = imputer_age.fit_transform(df[["Age"]])
print(f"'Age' imputed with median  : {imputer_age.statistics_[0]:.1f}")

# Strategy C — Impute 'Embarked' with the mode (most frequent port)
mode_embarked = df["Embarked"].mode()[0]
df["Embarked"].fillna(mode_embarked, inplace=True)
print(f"'Embarked' imputed with mode : '{mode_embarked}'")

# Strategy D — Impute 'Fare' with the median (precaution for test set)
df["Fare"].fillna(df["Fare"].median(), inplace=True)
print(f"'Fare' imputed with median : {df['Fare'].median():.2f}")

# Verification
print("\nRemaining missing values:")
print(df.isnull().sum())


---
## Exercise 3 — Feature Engineering

**New features created :**
- `FamilySize` = `SibSp` + `Parch` + 1  (passenger included)
- `IsAlone`    = 1 if `FamilySize == 1`, else 0
- `Title`      extracted from the `Name` column and grouped into major categories

Encoding is applied to `Title` here because it is a new categorical feature.  
`Sex` and `Embarked` are encoded in Exercise 6.


In [ ]:
# Feature 1 — FamilySize
# SibSp = number of siblings/spouses aboard
# Parch  = number of parents/children aboard
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
print("FamilySize distribution:")
print(df["FamilySize"].value_counts().sort_index())


In [ ]:
# Feature 2 — IsAlone (binary flag)
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
print("IsAlone value counts:")
print(df["IsAlone"].value_counts())


In [ ]:
# Feature 3 — Title extracted from the Name column
# The regex extracts the title (e.g. Mr, Mrs, Miss, Master...) from strings like
# "Braund, Mr. Owen Harris"
df["Title"] = df["Name"].str.extract(r",\s*([^.]+)\.")
print("\nRaw titles found:")
print(df["Title"].value_counts())


In [ ]:
# Group rare titles into broader categories to reduce cardinality
title_mapping = {
    "Mr"      : "Mr",
    "Miss"    : "Miss",
    "Mrs"     : "Mrs",
    "Master"  : "Master",
    # All rare titles → "Rare"
}
df["Title"] = df["Title"].map(title_mapping).fillna("Rare")

print("Grouped titles:")
print(df["Title"].value_counts())


In [ ]:
# Encode Title with Label Encoding (ordinal-like, compact)
le = LabelEncoder()
df["Title_encoded"] = le.fit_transform(df["Title"])

print("\nTitle encoding mapping:")
for cls, code_val in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {cls} -> {code_val}")

df[["Title", "Title_encoded"]].drop_duplicates().sort_values("Title_encoded")


---
## Exercise 4 — Outlier Detection and Handling

**Columns analysed :** `Fare` and `Age`

**Methods :**
- Visualisation : boxplot and histogram
- Detection     : IQR method (interquartile range)
- Treatment     : quantile capping (Winsorization) at the 98th percentile for `Fare`,
  log transformation for `Fare`, and IQR-based capping for `Age`


In [ ]:
# Visualise distributions before treatment
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

df["Fare"].plot(kind="hist", bins=50, ax=axes[0][0], title="Fare — histogram (before)")
df["Fare"].plot(kind="box",  ax=axes[0][1], title="Fare — boxplot (before)")
df["Age"].plot(kind="hist",  bins=40, ax=axes[1][0], title="Age — histogram (before)")
df["Age"].plot(kind="box",   ax=axes[1][1], title="Age — boxplot (before)")

plt.tight_layout()
plt.show()


In [ ]:
# IQR outlier detection
def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

for col in ["Fare", "Age"]:
    lower, upper = iqr_bounds(df[col])
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col:6s} — lower bound: {lower:.2f}, upper bound: {upper:.2f}, outliers: {n_outliers}")


In [ ]:
# Treatment 1 — Quantile capping for Fare (Winsorization at 98th percentile)
# Extreme fares (e.g. 512) are capped rather than removed to preserve the row.
cap_98 = df["Fare"].quantile(0.98)
print(f"Fare 98th percentile : {cap_98:.2f}")
df["Fare"] = df["Fare"].clip(upper=cap_98)
print(f"Fare max after capping : {df['Fare'].max():.2f}")


In [ ]:
# Treatment 2 — Log transformation for Fare to reduce right skew
# log1p is used to handle zero fares (log(0) is undefined).
df["Fare_log"] = np.log1p(df["Fare"])
print("Fare_log stats:")
print(df["Fare_log"].describe().round(3))


In [ ]:
# Treatment 3 — IQR capping for Age
lower_age, upper_age = iqr_bounds(df["Age"])
df["Age"] = df["Age"].clip(lower=max(0, lower_age), upper=upper_age)
print(f"Age clipped to [{max(0, lower_age):.1f}, {upper_age:.1f}]")


In [ ]:
# Compare before / after (Fare)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["Fare"].plot(kind="hist", bins=50, ax=axes[0], title="Fare — after capping")
df["Fare_log"].plot(kind="hist", bins=50, ax=axes[1], title="Fare_log — after log transform")
plt.tight_layout()
plt.show()


---
## Exercise 5 — Data Standardization and Normalization

**After outlier treatment**, we scale numerical features :

| Feature      | Scaler       | Reason                                    |
|--------------|--------------|-------------------------------------------|
| `Age`        | StandardScaler | Approximately normal after IQR capping  |
| `Fare_log`   | MinMaxScaler   | Still right-skewed; bounded after log   |
| `FamilySize` | MinMaxScaler   | Discrete, bounded [1, 11]               |


In [ ]:
# StandardScaler — Age (approximately Gaussian after outlier treatment)
scaler_std = StandardScaler()
df["Age_scaled"] = scaler_std.fit_transform(df[["Age"]])
print("Age_scaled — mean:", df["Age_scaled"].mean().round(4),
      " std:", df["Age_scaled"].std().round(4))

# MinMaxScaler — Fare_log (skewed, bounded)
scaler_mm = MinMaxScaler()
df["Fare_scaled"] = scaler_mm.fit_transform(df[["Fare_log"]])
print("Fare_scaled — min:", df["Fare_scaled"].min().round(4),
      " max:", df["Fare_scaled"].max().round(4))

# MinMaxScaler — FamilySize (discrete, bounded)
df["FamilySize_scaled"] = scaler_mm.fit_transform(df[["FamilySize"]])
print("FamilySize_scaled — min:", df["FamilySize_scaled"].min().round(4),
      " max:", df["FamilySize_scaled"].max().round(4))


In [ ]:
# Visualise scaled features
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df["Age_scaled"].plot(kind="hist", bins=30, ax=axes[0], title="Age_scaled (StandardScaler)")
df["Fare_scaled"].plot(kind="hist", bins=30, ax=axes[1], title="Fare_scaled (MinMaxScaler)")
df["FamilySize_scaled"].plot(kind="hist", bins=15, ax=axes[2], title="FamilySize_scaled (MinMaxScaler)")
plt.tight_layout()
plt.show()


---
## Exercise 6 — Feature Encoding

**Remaining categorical columns :** `Sex`, `Embarked`

- `Sex`      : binary, label-encoded (male=1, female=0)
- `Embarked` : nominal with 3 values, one-hot encoded (drop_first=True to avoid multicollinearity)

`Title` was already encoded in Exercise 3 and is not re-encoded here.


In [ ]:
# Label encoding for Sex (binary feature)
le_sex = LabelEncoder()
df["Sex_encoded"] = le_sex.fit_transform(df["Sex"])
print("Sex encoding:", dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_))))


In [ ]:
# One-hot encoding for Embarked (nominal, 3 categories: C, Q, S)
# drop_first=True removes one dummy column to avoid the dummy variable trap.
embarked_dummies = pd.get_dummies(df["Embarked"], prefix="Embarked", drop_first=True)
print("One-hot encoded Embarked columns:", list(embarked_dummies.columns))
print(embarked_dummies.head())

# Merge into the main dataframe
df = pd.concat([df, embarked_dummies], axis=1)


In [ ]:
# Summary of encoded columns
print("Encoded columns added to df:")
print(df[["Sex", "Sex_encoded", "Embarked", "Embarked_Q", "Embarked_S"]].head(8))


---
## Exercise 7 — Age Feature Transformation

**Objective :** Create life-stage bins from the `Age` column and one-hot encode them.

**Bins defined :**
- [0, 12)   → Child
- [12, 18)  → Teenager
- [18, 60)  → Adult
- [60, 100] → Senior


In [ ]:
# Step 1 — Create age bins with pd.cut()
# We use the original (clipped) Age column, not the scaled version.
bins   = [0, 12, 18, 60, 100]
labels = ["Child", "Teenager", "Adult", "Senior"]

df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False)

print("AgeGroup distribution:")
print(df["AgeGroup"].value_counts().sort_index())


In [ ]:
# Visualise age groups
df["AgeGroup"].value_counts().sort_index().plot(
    kind="bar", color=["#4e79a7", "#f28e2b", "#59a14f", "#e15759"],
    figsize=(7, 4), title="Passenger count per age group", rot=0
)
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# Step 2 — One-hot encoding with get_dummies()
# drop_first=False is used here to keep all groups explicit for interpretability.
age_dummies = pd.get_dummies(df["AgeGroup"], prefix="AgeGroup")
print("One-hot encoded AgeGroup columns:", list(age_dummies.columns))
print(age_dummies.head())

# Merge into the main dataframe
df = pd.concat([df, age_dummies], axis=1)


In [ ]:
# Final dataset overview
print("Final dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nSample rows:")
df.head()


In [ ]:
# Final check — no remaining missing values in key columns
key_cols = ["Age", "Fare", "Embarked", "Sex", "FamilySize",
            "Title_encoded", "Sex_encoded", "Age_scaled", "Fare_scaled"]
print("Missing values in key columns:")
print(df[key_cols].isnull().sum())


---
## Summary — Preprocessing Pipeline

| Step | Action | Columns affected |
|------|--------|-----------------|
| 1 | Remove duplicates | All |
| 2 | Drop `Cabin` (77 % missing) | Cabin |
| 2 | Median imputation | Age, Fare |
| 2 | Mode imputation | Embarked |
| 3 | Create `FamilySize`, `IsAlone` | SibSp, Parch |
| 3 | Extract and encode `Title` | Name |
| 4 | IQR capping | Age |
| 4 | Quantile capping (98th pct) + log transform | Fare |
| 5 | StandardScaler | Age |
| 5 | MinMaxScaler | Fare_log, FamilySize |
| 6 | Label encoding | Sex |
| 6 | One-hot encoding | Embarked |
| 7 | Age bins + one-hot encoding | AgeGroup |
